**PROYECTO**       : GESTION INTEGRAL PASIVOS  
**NOMBRE**         : nb_fct_saldo_cuenta_pasivo.ipynb  
**TABLA DESTINO**  : mb_gold_prod.pasivos.fct_saldo_cuenta_pasivo  
**TABLA FUENTE**   : mb_silver_prod.mmff.h_saldo_cuenta  
**OBJETIVO**       : Poblar el hecho de saldos de cuentas pasivas  
**TIPO**           : PYTHON  
**REPROCESABLE**   : SI  
**OBSERVACION**    : NA  
**SCHEDULER**      : NA  
**JOB**            : NA
| VERSION | DESARROLLADOR | PROVEEDOR | PO | FECHA | DESCRIPCION |
|---------|---------------|-----------|----|-------|-------------|
| 1.0 | Kevin Ponce | MIBANCO | Enith Rodriguez | 2026-09-11 | Creacion de proceso |

## 1. Librerias y dependencias

In [0]:
import logging
import time

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

## 2. Parametros de entrada

In [0]:
dbutils.widgets.text("ambiente", "dev")
dbutils.widgets.text("fechaproceso", "")

var_ambiente = dbutils.widgets.get("ambiente")
var_fechaproceso = dbutils.widgets.get("fechaproceso")

logger = logging.getLogger("FCT_SALDO_CUENTA_PASIVO")
logger.setLevel(logging.INFO)
logger.info("Inicio del proceso. ambiente=%s fechaproceso=%s", var_ambiente, var_fechaproceso)

## 3. Constantes y variables

In [0]:
TBL_SALDO_SRC = f"mb_silver_{var_ambiente}.mmff.h_saldo_cuenta"
TBL_AGENCIA_SRC = f"mb_silver_{var_ambiente}.mmff.m_agencia"
TBL_SALDO_FIN = f"mb_gold_{var_ambiente}.pasivos.fct_saldo_cuenta_pasivo"

EST_VIGENTE = "VIGENTE"

## 4. Funciones de transformacion

In [0]:
def read_saldos(tabla):
    """Lee la tabla de saldos de cuenta."""
    return spark.table(tabla).select("*")


def add_saldo_promedio(df_saldos):
    """Calcula el saldo promedio por cuenta."""
    ventana = Window.partitionBy("cod_cuenta")
    return df_saldos.withColumn(
        "mto_saldo_promedio", F.avg(F.col("mto_saldo")).over(ventana)
    )

## 5. Logica principal

In [0]:
ini_proceso = time.perf_counter()

df_saldos = read_saldos(TBL_SALDO_SRC)
df_agencias = spark.table(TBL_AGENCIA_SRC)

df_join = df_saldos.join(
    df_agencias, df_saldos["cod_agencia"] == df_agencias["cod_agencia"], "left"
)

df_enriquecido = add_saldo_promedio(df_join)

df_final = (
    df_enriquecido
    .filter(F.col("fec_proceso") == var_fechaproceso)
    .filter(F.col("est_cuenta") == EST_VIGENTE)
)

In [0]:
ctd_registros = len(df_final.collect())
print("Registros a escribir: " + str(ctd_registros))

## 6. Escritura

In [0]:
(
    df_final
    .repartition(200)
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(TBL_SALDO_FIN)
)

logger.info("Fin del proceso. Duracion: %.2f segundos", time.perf_counter() - ini_proceso)